In [ ]:
# ─── 1. Install ───────────────────────────────────────────────────────────────
!pip uninstall -y peft accelerate transformers -q
!pip install -q \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  safetensors==0.4.2 \
  peft==0.7.1 \
  scikit-learn \
  Pillow \
  numpy==1.26.4

import os
os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.3/168.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.4/166.4 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1

In [ ]:
# ─── 2. Mount Drive ───────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/CLIP_ViT_B32_Adapter_IFND"
import os
os.makedirs(SAVE_DIR, exist_ok=True)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# ─── 3. Imports ───────────────────────────────────────────────────────────────
import torch
import numpy as np
import functools
import time
from PIL import Image
from torch import nn
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score
from transformers import CLIPProcessor, CLIPModel, TrainingArguments, Trainer, EarlyStoppingCallback
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers.trainer_utils import get_last_checkpoint
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ─── 4. Load dataset ──────────────────────────────────────────────────────────
print("⏳ Loading dataset...")
hf_dataset = load_dataset("Nhat243/IFND-multimodal")
print(hf_dataset)

# ─── 5. Preprocess ────────────────────────────────────────────────────────────
model_name = "openai/clip-vit-base-patch32"
processor  = CLIPProcessor.from_pretrained(model_name)

def preprocess(example):
    try:
        image = example['image'].convert("RGB")
    except:
        image = Image.new("RGB", (224, 224), (128, 128, 128))
    inputs = processor(
        text=str(example['text']),
        images=image,
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )
    return {
        "input_ids":      inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values":   inputs["pixel_values"][0],
        "labels":         int(example["label"])
    }

print("⏳ Preprocessing dataset...")
encoded_dataset = hf_dataset.map(
    preprocess,
    remove_columns=hf_dataset["train"].column_names,
    desc="Preprocessing"
)
encoded_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "pixel_values", "labels"]
)

train_dataset = encoded_dataset["train"]
val_dataset   = encoded_dataset["validation"]
test_dataset  = encoded_dataset["test"]
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# ─── 6. Adapter Module ────────────────────────────────────────────────────────
class Adapter(nn.Module):
    def __init__(self, dim, bottleneck=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, bottleneck),
            nn.GELU(),
            nn.Linear(bottleneck, dim)
        )

    def forward(self, x):
        return x + self.net(x)  # residual connection

class CLIPWithAdapter(nn.Module):
    def __init__(self, model_name, num_labels=2, bottleneck=64):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)

        # Freeze toàn bộ CLIP
        for param in self.clip.parameters():
            param.requires_grad = False

        # Thêm Adapter vào mỗi transformer layer của text encoder
        text_dim = self.clip.config.text_config.hidden_size
        img_dim  = self.clip.config.vision_config.hidden_size

        self.text_adapters = nn.ModuleList([
            Adapter(text_dim, bottleneck)
            for _ in range(self.clip.config.text_config.num_hidden_layers)
        ])
        self.img_adapters = nn.ModuleList([
            Adapter(img_dim, bottleneck)
            for _ in range(self.clip.config.vision_config.num_hidden_layers)
        ])

        # Classifier head
        self.classifier = nn.Linear(self.clip.config.projection_dim * 2, num_labels)

    def forward(self, input_ids=None, attention_mask=None,
                pixel_values=None, labels=None):
        # Text encoding với adapter
        text_outputs = self.clip.text_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
            return_dict=True
        )
        # Apply text adapters trên hidden states
        text_hidden = text_outputs.last_hidden_state
        for adapter in self.text_adapters:
            text_hidden = adapter(text_hidden)
        text_embeds = self.clip.text_projection(
            text_hidden[torch.arange(text_hidden.shape[0]),
                        input_ids.argmax(dim=-1)]
        )
        text_embeds = text_embeds / text_embeds.norm(dim=-1, keepdim=True)

        # Image encoding với adapter
        vision_outputs = self.clip.vision_model(
            pixel_values=pixel_values,
            output_hidden_states=True,
            return_dict=True
        )
        img_hidden = vision_outputs.last_hidden_state
        for adapter in self.img_adapters:
            img_hidden = adapter(img_hidden)
        img_embeds = self.clip.visual_projection(img_hidden[:, 0, :])
        img_embeds = img_embeds / img_embeds.norm(dim=-1, keepdim=True)

        # Fusion + classify
        fused  = torch.cat([text_embeds, img_embeds], dim=1)
        logits = self.classifier(fused)
        loss   = nn.CrossEntropyLoss()(logits, labels) if labels is not None else None
        return SequenceClassifierOutput(loss=loss, logits=logits)

torch.cuda.empty_cache()
import gc; gc.collect()

model = CLIPWithAdapter(model_name).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,} ({trainable_params/total_params*100:.2f}%)")

# ─── 7. Metrics ───────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds  = np.argmax(probs, axis=1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    return {
        "accuracy":  accuracy_score(labels, preds),
        "precision": p,
        "recall":    r,
        "f1":        f1,
        "auc":       roc_auc_score(labels, probs[:, 1])
    }

# ─── 8. Training Args ─────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=0,
    report_to="none"
)

# ─── 9. Trainer ───────────────────────────────────────────────────────────────
torch.load = functools.partial(torch.load, weights_only=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

last_checkpoint = get_last_checkpoint(SAVE_DIR)
if last_checkpoint:
    print(f"▶️ Resuming from: {last_checkpoint}")
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("🆕 Training from scratch")
    trainer.train()

trainer.save_model(SAVE_DIR)
processor.save_pretrained(SAVE_DIR)
print("✅ Saved to:", SAVE_DIR)

# ─── 10. Latency + VRAM ───────────────────────────────────────────────────────
model.eval()
dummy_image = Image.new("RGB", (224, 224), (128, 128, 128))
dummy_text  = "This is a sample news headline for inference measurement."
inputs = processor(
    text=dummy_text, images=dummy_image,
    truncation=True, padding="max_length",
    max_length=77, return_tensors="pt"
)
input_ids      = inputs["input_ids"].to(device)
attention_mask = inputs["attention_mask"].to(device)
pixel_values   = inputs["pixel_values"].to(device)

if device == "cuda":
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

with torch.no_grad():
    for _ in range(10):
        model(input_ids=input_ids, attention_mask=attention_mask,
              pixel_values=pixel_values)

latencies = []
with torch.no_grad():
    for _ in range(100):
        if device == "cuda": torch.cuda.synchronize()
        t0 = time.perf_counter()
        model(input_ids=input_ids, attention_mask=attention_mask,
              pixel_values=pixel_values)
        if device == "cuda": torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000)

latencies = np.array(latencies)
vram = torch.cuda.max_memory_allocated() / 1024**2 if device == "cuda" else 0

print(f"\n{'='*40}")
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")
print(f"Latency median  : {np.median(latencies):.2f} ms")
print(f"VRAM (peak)     : {vram:.1f} MB")
print(f"{'='*40}")

# ─── 11. Test Evaluation ──────────────────────────────────────────────────────
print("\n📊 Evaluating on test set...")
test_results = trainer.predict(test_dataset)
logits = test_results.predictions
labels = test_results.label_ids
probs  = torch.softmax(torch.tensor(logits), dim=1).numpy()
preds  = np.argmax(probs, axis=1)

p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
print(f"\n{'='*40}")
print(f"Test Accuracy : {accuracy_score(labels, preds)*100:.2f}%")
print(f"Test Precision: {p*100:.2f}%")
print(f"Test Recall   : {r*100:.2f}%")
print(f"Test F1       : {f1*100:.2f}%")
print(f"Test AUC      : {roc_auc_score(labels, probs[:, 1]):.5f}")
print(f"{'='*40}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda
⏳ Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/8416 [00:00<?, ? examples/s]

...

Total params    : 153,262,339
Trainable params: 1,985,026 (1.30%)
🆕 Training from scratch


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.503800,0.316020,0.961977,0.952719,1.000000,0.975787,0.993471
2,0.271700,0.233440,0.981939,0.980464,0.996278,0.988308,0.993749
3,0.215000,0.196378,0.983840,0.984049,0.995037,0.989513,0.993983
4,0.181500,0.179126,0.982890,0.982843,0.995037,0.988903,0.994230
5,0.166000,0.170942,0.983840,0.984049,0.995037,0.989513,0.994843


✅ Saved to: /content/drive/MyDrive/CLIP_ViT_B32_Adapter_IFND

Total params    : 153,262,339
Trainable params: 1,985,026
Latency median  : 37.94 ms
VRAM (peak)     : 631.1 MB

📊 Evaluating on test set...



Test Accuracy : 98.58%
Test Precision: 98.53%
Test Recall   : 99.63%
Test F1       : 99.08%
Test AUC      : 0.99266
